# Social Media Engagement Analytics Using Python
**Module-End Assignment 5 - Data Analytics (Python for Data Analysis)**

Dataset: `social_media_engagement_5000.csv` (5,000 posts). The notebook loads it directly from GitHub, falls back to a manual upload if the download fails, and **every insight in the final summary is computed from the data**, not typed by hand.

Sections: 1 Import and Setup, 2 Cleaning, 3 Exploration, 4 Wrangling, 5 Statistics, 6 Visualisation (matplotlib, seaborn, plotly), 7 Insights and one-page summary.

## Task 1 - Data Import and Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
sns.set_theme(style="whitegrid")

URL = "https://raw.githubusercontent.com/GeethaGunasekaran1/Dataset_rep/main/social_media_engagement_5000.csv"
try:
    df = pd.read_csv(URL)
except Exception as e:
    print("Download failed, please upload the CSV manually:", e)
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(next(iter(up)))

raw = df.copy()          # untouched copy for before/after comparisons
print("Loaded shape:", df.shape)
df.head()

In [ ]:
# Check data types, then convert date columns to datetime
print(df.dtypes)

# Column names are standardised (lower case, underscores) so the code is robust to small naming differences
df.columns = [re.sub(r"\W+", "_", c.strip().lower()).strip("_") for c in df.columns]

def pick(*names, contains=None):
    """Return the first existing column from candidates, else a column containing a keyword."""
    for n in names:
        if n in df.columns:
            return n
    if contains:
        for c in df.columns:
            if contains in c:
                return c
    return None

C = dict(
    date   = pick("posted_at", "post_date", "date", "timestamp", contains="date") or pick(contains="time"),
    likes  = pick("likes"), comments = pick("comments"), shares = pick("shares"),
    watch  = pick("watch_time_sec", "watch_time", contains="watch"),
    imp    = pick("impression_count", "impressions", contains="impress"),
    er     = pick("engagement_rate", contains="engagement_rate"),
    fol    = pick("follower_count", "followers", contains="follower"),
    age    = pick("age"), gender = pick("gender"), country = pick("country"),
    ptype  = pick("post_type", contains="type") , cat = pick("post_category", "category", contains="categ"),
    ver    = pick("is_verified", "verified", contains="verif"),
    dev    = pick("device_type", "device", contains="device"),
    sent   = pick("sentiment", "sentiment_label", contains="sentiment"),
    tags   = pick("hashtags", contains="hashtag"),
)
if C["ptype"] == C["dev"]:  # 'contains type' could clash with device_type
    C["ptype"] = pick("post_type")
print({k: v for k, v in C.items()})

In [ ]:
# Convert date column to datetime
df[C["date"]] = pd.to_datetime(df[C["date"]], errors="coerce")

# Numeric columns that must be numeric
num_cols = [C[k] for k in ["likes","comments","shares","watch","imp","er","fol","age"] if C[k]]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
print(df.dtypes)

In [ ]:
# Basic NumPy array operations
likes_arr = df[C["likes"]].to_numpy(dtype=float)
print("Array shape      :", likes_arr.shape)
print("Mean / median    :", np.nanmean(likes_arr).round(2), "/", np.nanmedian(likes_arr).round(2))
print("Min / max        :", np.nanmin(likes_arr), "/", np.nanmax(likes_arr))
print("90th percentile  :", np.nanpercentile(likes_arr, 90).round(2))
print("Likes standardised (first 5 z-scores):", ((likes_arr - np.nanmean(likes_arr)) / np.nanstd(likes_arr))[:5].round(3))
print("Posts with likes above mean:", int(np.sum(likes_arr > np.nanmean(likes_arr))))

## Task 2 - Data Cleaning

### 2.1 Missing values

In [ ]:
miss = df.isnull().sum().to_frame("missing")
miss["pct"] = (miss["missing"] / len(df) * 100).round(2)
miss[miss["missing"] > 0]

In [ ]:
before_missing = int(df.isna().sum().sum())

# Rule: numeric -> median (robust to skew); categorical -> mode; rows with no date or no id cannot be used -> dropped
df = df.dropna(subset=[C["date"]])
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())
cat_cols = [C[k] for k in ["gender","country","ptype","cat","ver","dev","sent"] if C[k]]
for c in cat_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].mode()[0])
# Alternative methods required by the brief (forward / backward fill) shown on a sorted time series:
df = df.sort_values(C["date"]).reset_index(drop=True)
df[num_cols] = df[num_cols].ffill().bfill()

print("Missing before:", before_missing, "| after:", int(df.isna().sum().sum()))
# Any values still missing are blank hashtag fields = posts with no hashtags. That is a valid value (count = 0), so it is kept, not imputed.
print("Still missing by column:", df.isna().sum()[df.isna().sum() > 0].to_dict())

### 2.2 Duplicates

In [ ]:
id_col = pick("post_id", contains="post_id")
print("Fully duplicated rows :", df.duplicated().sum())
if id_col:
    print("Duplicated post ids   :", df.duplicated(subset=[id_col]).sum())
n0 = len(df)
df = df.drop_duplicates()
if id_col:
    df = df.drop_duplicates(subset=[id_col], keep="first")
df = df.reset_index(drop=True)
print(f"Rows removed: {n0 - len(df)} | rows now: {len(df)}")

### 2.3 Formatting: categories, data types, unrealistic values

In [ ]:
# Standardise text categories (trim, unify case, map common variants)
for c in cat_cols:
    if c != C["ver"]:
        # keep short upper-case codes such as USA / UK intact, title-case everything else
        df[c] = df[c].astype(str).str.strip().apply(lambda v: v if (v.isupper() and len(v) <= 3) else v.title())

gender_map = {"M": "Male", "Male": "Male", "F": "Female", "Female": "Female",
              "Other": "Other", "Non-Binary": "Other", "Nonbinary": "Other", "Nan": "Unknown"}
df[C["gender"]] = df[C["gender"]].replace(gender_map)

# Verified flag -> proper boolean
df[C["ver"]] = (df[C["ver"]].astype(str).str.strip().str.lower()
                .map({"true": True, "1": True, "yes": True, "y": True, "false": False, "0": False, "no": False, "n": False})
                .fillna(False).astype(bool))

# Sentiment labels cleaned
df[C["sent"]] = df[C["sent"]].replace({"Pos": "Positive", "Neg": "Negative", "Neu": "Neutral"})

for c in cat_cols:
    if c != C["ver"]:
        print(c, "->", sorted(df[c].unique())[:12])

In [ ]:
# Unrealistic values: negatives are impossible; extreme outliers are capped at the 99.5th percentile (winsorising)
count_cols = [C[k] for k in ["likes","comments","shares","watch","imp","fol"] if C[k]]
neg_before = {c: int((df[c] < 0).sum()) for c in count_cols}
print("Negative values found:", neg_before)
for c in count_cols:
    df[c] = df[c].clip(lower=0)

out_before = {}
for c in [C["likes"], C["comments"], C["shares"]]:
    cap = df[c].quantile(0.995)
    out_before[c] = int((df[c] > cap).sum())
    df[c] = df[c].clip(upper=cap)
print("Values capped at 99.5th percentile:", out_before)

# Age must be realistic (13-100); invalid ages replaced by the median
bad_age = ~df[C["age"]].between(13, 100)
print("Unrealistic ages fixed:", int(bad_age.sum()))
df.loc[bad_age, C["age"]] = df[C["age"]].median()

# Engagement rate should be a non-negative number
df[C["er"]] = df[C["er"]].clip(lower=0)

### 2.4 Feature cleaning

In [ ]:
def count_hashtags(x):
    if pd.isna(x): return 0
    return len(re.findall(r"#\w+", str(x)))

if C["tags"]:
    sample = df[C["tags"]].astype(str).head(3).tolist()
    df["hashtag_count"] = df[C["tags"]].apply(count_hashtags)
    # if hashtags were stored without '#', fall back to counting separators
    if df["hashtag_count"].sum() == 0:
        df["hashtag_count"] = df[C["tags"]].fillna("").astype(str).apply(lambda s: len([t for t in re.split(r"[,; ]+", s) if t]))
    print("Sample raw values:", sample)
print(df["hashtag_count"].value_counts().sort_index())
print("Sentiment labels now:", df[C["sent"]].unique())

## Task 3 - Data Exploration with Pandas

In [ ]:
display(df.head()); display(df.tail())
print("Shape:", df.shape); print("Columns:", list(df.columns))

In [ ]:
df.info()
print(); print(df.dtypes)

In [ ]:
df.describe().T

In [ ]:
for c in [C["ptype"], C["cat"], C["country"], C["gender"], C["dev"], C["sent"]]:
    print(f"--- {c}  (unique: {df[c].nunique()})")
    print(df[c].value_counts().head(10).to_string()); print()
print("Unique post types:", df[C["ptype"]].unique())

In [ ]:
num_df = df.select_dtypes(include="number")
corr = num_df.corr()
corr

In [ ]:
display(df.groupby(C["ptype"])[[C["likes"], C["comments"], C["shares"], C["er"]]].mean().round(2))
display(df.groupby(C["country"])[C["imp"]].mean().round(1).sort_values(ascending=False).head(10).to_frame("avg_impressions"))

## Task 4 - Data Wrangling

In [ ]:
# Engagement score and log transforms
df["engagement_score"] = df[C["likes"]] + 2 * df[C["comments"]] + 3 * df[C["shares"]]   # weights: deeper actions count more
for c in [C["likes"], C["imp"], C["fol"]]:
    df["log_" + c] = np.log1p(df[c])

# Time features
df["post_hour"] = df[C["date"]].dt.hour
df["post_date"] = df[C["date"]].dt.date
df["month"] = df[C["date"]].dt.to_period("M").astype(str)
df["age_group"] = pd.cut(df[C["age"]], bins=[12, 17, 24, 34, 44, 54, 100],
                         labels=["13-17", "18-24", "25-34", "35-44", "45-54", "55+"])
df["time_of_day"] = pd.cut(df["post_hour"], bins=[-1, 5, 11, 17, 23],
                           labels=["Night (0-5)", "Morning (6-11)", "Afternoon (12-17)", "Evening (18-23)"])
df[[C["likes"], C["comments"], C["shares"], "engagement_score", "hashtag_count", "age_group", "time_of_day"]].head()

In [ ]:
# concat / merge demonstration: build lookup tables and join them back
country_avg = df.groupby(C["country"], as_index=False)[C["er"]].mean().rename(columns={C["er"]: "country_avg_er"})
type_avg    = df.groupby(C["ptype"], as_index=False)["engagement_score"].mean().rename(columns={"engagement_score": "type_avg_score"})
df = df.merge(country_avg, on=C["country"], how="left").merge(type_avg, on=C["ptype"], how="left")
df["er_vs_country_avg"] = df[C["er"]] - df["country_avg_er"]

top_half = df[df["engagement_score"] >= df["engagement_score"].median()].assign(segment="High engagement")
low_half = df[df["engagement_score"] <  df["engagement_score"].median()].assign(segment="Low engagement")
stacked = pd.concat([top_half, low_half], axis=0)
print("concat check:", stacked.shape, "| merge columns added:", ["country_avg_er", "type_avg_score"])

In [ ]:
metrics = [C["likes"], C["comments"], C["shares"], C["er"], "engagement_score"]
by_type    = df.groupby(C["ptype"])[metrics].mean().round(2).sort_values("engagement_score", ascending=False)
by_country = df.groupby(C["country"])[metrics].mean().round(2).sort_values(C["er"], ascending=False)
by_sent    = df.groupby(C["sent"])[metrics + [C["fol"], C["watch"]]].mean().round(2).sort_values("engagement_score", ascending=False)
display(by_type); display(by_country); display(by_sent)

## Task 5 - Statistical Analysis

In [ ]:
stat_cols = [c for c in [C["likes"], C["comments"], C["shares"], C["watch"], C["er"], C["fol"]] if c]
rows = {}
for c in stat_cols:
    s = df[c]
    rows[c] = {"mean": s.mean(), "median": s.median(), "mode": s.mode().iloc[0],
               "std": s.std(), "variance": s.var(),
               "p25": s.quantile(.25), "p50": s.quantile(.50), "p75": s.quantile(.75), "p90": s.quantile(.90), "p95": s.quantile(.95),
               "skewness": s.skew(), "kurtosis": s.kurt()}
stats_table = pd.DataFrame(rows).T
stats_table

## Task 6 - Data Visualisation (16 plots)

### Matplotlib (6)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df[C["imp"]], df[C["likes"]], s=10, alpha=.35, color="#2a6f97")
ax.set_xlabel("Impressions"); ax.set_ylabel("Likes"); ax.set_title("1. Likes vs Impressions")
plt.show()
r = df[[C["imp"], C["likes"]]].corr().iloc[0, 1]; print(f"Correlation likes vs impressions: {r:.3f}")

In [ ]:
daily = df.groupby("post_date")["engagement_score"].mean()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(pd.to_datetime(daily.index), daily.values, lw=.8, color="#bc4749", alpha=.5, label="Daily")
ax.plot(pd.to_datetime(daily.index), daily.rolling(30, min_periods=5).mean().values, lw=2, color="#1d3557", label="30-day average")
ax.set_title("2. Daily Average Engagement Score"); ax.set_ylabel("Engagement score"); ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
df[C["cat"]].value_counts().plot.bar(ax=ax, color="#52796f")
ax.set_title("3. Posts by Category"); ax.set_ylabel("Number of posts"); plt.xticks(rotation=40, ha="right")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
g = df[C["gender"]].value_counts()
ax.pie(g.values, labels=g.index, autopct="%1.1f%%", startangle=90, colors=sns.color_palette("pastel"))
ax.set_title("4. Gender Distribution"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(df[C["age"]], bins=25, color="#6a4c93", edgecolor="white")
ax.set_title("5. Age Distribution"); ax.set_xlabel("Age"); ax.set_ylabel("Users"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.boxplot(df[C["er"]].dropna(), vert=True, patch_artist=True, boxprops=dict(facecolor="#9ecae1"))
ax.set_title("6. Engagement Rate (box plot)"); ax.set_ylabel("Engagement rate"); plt.show()

### Seaborn (6)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
order = df[C["ptype"]].value_counts().index
sns.countplot(data=df, x=C["ptype"], order=order, palette="viridis", ax=ax)
ax.set_title("7. Count of Posts by Post Type"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=df, x=C["cat"], y=C["likes"], palette="mako", errorbar=None, ax=ax,
            order=df.groupby(C["cat"])[C["likes"]].mean().sort_values(ascending=False).index)
ax.set_title("8. Average Likes by Category"); plt.xticks(rotation=40, ha="right"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=df, x=C["sent"], y=C["fol"], palette="Set2", ax=ax, cut=0)
ax.set_title("9. Followers by Sentiment (violin)"); plt.show()

In [ ]:
pair_cols = [C["likes"], C["comments"], C["shares"], C["imp"], C["er"]]
sns.pairplot(df.sample(min(1000, len(df)), random_state=1)[pair_cols], corner=True, plot_kws=dict(s=10, alpha=.4))
plt.suptitle("10. Pair Plot of Numeric Features (1,000-post sample)", y=1.02); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(num_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, annot_kws={"size": 7})
ax.set_title("11. Correlation Matrix"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.swarmplot(data=df.sample(min(600, len(df)), random_state=2), x=C["dev"], y=C["er"], size=2.5, palette="Set1", ax=ax)
ax.set_title("12. Engagement Rate by Device (swarm, 600-post sample)"); plt.show()

### Plotly - interactive (4)

In [ ]:
monthly = df.groupby("month", as_index=False)["engagement_score"].mean()
px.line(monthly, x="month", y="engagement_score", markers=True, title="13. Monthly Average Engagement Score (interactive)").show()

In [ ]:
t = df.groupby(C["ptype"], as_index=False)[C["er"]].mean()
px.bar(t, x=C["ptype"], y=C["er"], color=C["ptype"], title="14. Average Engagement Rate by Post Type (interactive)").show()

In [ ]:
px.scatter(df.sample(min(1500, len(df)), random_state=3), x=C["imp"], y=C["likes"], color=C["sent"],
           hover_data=[C["country"], C["ptype"]], title="15. Likes vs Impressions by Sentiment (interactive)").show()

In [ ]:
b = df.groupby(C["country"], as_index=False).agg(avg_er=(C["er"], "mean"), avg_imp=(C["imp"], "mean"), posts=(C["er"], "size"))
px.scatter(b, x="avg_imp", y="avg_er", size="posts", color=C["country"], title="16. Country Bubble Chart: impressions vs engagement rate").show()

## Final Insights (all figures computed from the cleaned data)

In [ ]:
def best(group, metric, df_=None):
    d = (df_ if df_ is not None else df).groupby(group)[metric].mean().sort_values(ascending=False)
    return d.index[0], d.iloc[0], d.index[-1], d.iloc[-1]

L = {}   # insight lines collected for the one-page summary
t_hi, t_hv, t_lo, t_lv = best(C["ptype"], "engagement_score")
L["Content performance"] = [
    f"Post type with highest engagement score: {t_hi} ({t_hv:,.1f}); lowest: {t_lo} ({t_lv:,.1f}).",
]
c_hi, c_hv, c_lo, c_lv = best(C["cat"], "engagement_score")
L["Content performance"].append(f"Best-performing category: {c_hi} ({c_hv:,.1f}); weakest: {c_lo} ({c_lv:,.1f}).")
k_hi, k_hv, k_lo, k_lv = best(C["country"], C["er"])
top3 = df.groupby(C["country"])[C["er"]].mean().sort_values(ascending=False).head(3)
L["Content performance"].append("Countries with highest average engagement rate: " + ", ".join(f"{i} ({v:.3f})" for i, v in top3.items()) + ".")

age_r = df[[C["age"], "engagement_score"]].corr().iloc[0, 1]
ag = df.groupby("age_group", observed=True)[C["er"]].mean()
ver = df.groupby(C["ver"])[[C["er"], "engagement_score", C["likes"]]].mean()
L["User trends"] = [
    f"Age vs engagement score correlation: {age_r:.3f}; age group with highest engagement rate: {ag.idxmax()} ({ag.max():.3f}), lowest: {ag.idxmin()} ({ag.min():.3f}).",
]
if True in ver.index and False in ver.index:
    d = (ver.loc[True, "engagement_score"] / ver.loc[False, "engagement_score"] - 1) * 100
    L["User trends"].append(f"Verified accounts average an engagement score of {ver.loc[True,'engagement_score']:,.1f} vs {ver.loc[False,'engagement_score']:,.1f} for unverified ({d:+.1f}%); engagement rate {ver.loc[True,C['er']]:.3f} vs {ver.loc[False,C['er']]:.3f}.")

tod = df.groupby("time_of_day", observed=True)[C["imp"]].mean()
hr = df.groupby("post_hour")[C["imp"]].mean()
dv = df.groupby(C["dev"])[C["watch"]].mean().sort_values(ascending=False)
L["Behavioural insights"] = [
    f"Best time of day for impressions: {tod.idxmax()} ({tod.max():,.0f} avg); single best hour: {int(hr.idxmax())}:00 ({hr.max():,.0f}); weakest: {int(hr.idxmin())}:00 ({hr.min():,.0f}).",
    f"Device impact on watch time: highest on {dv.index[0]} ({dv.iloc[0]:.1f}s), lowest on {dv.index[-1]} ({dv.iloc[-1]:.1f}s); spread {(dv.iloc[0]/dv.iloc[-1]-1)*100:.1f}%.",
]

sg = df.groupby(C["sent"])[[C["er"], "engagement_score", C["likes"], C["comments"], C["shares"], C["watch"]]].mean()
sg_n = df[C["sent"]].value_counts()
best_s = sg["engagement_score"].idxmax()
L["Sentiment"] = [
    f"Best-performing sentiment: {best_s} (engagement score {sg.loc[best_s,'engagement_score']:,.1f}).",
    "Sentiment breakdown (avg engagement score / share of posts): " + "; ".join(f"{s}: {sg.loc[s,'engagement_score']:,.1f} / {sg_n[s]/len(df)*100:.1f}%" for s in sg.index) + ".",
]
neg = [s for s in sg.index if s.lower() in ("negative", "neutral")]
for s in neg:
    L["Sentiment"].append(f"{s} posts: {sg.loc[s,'engagement_score']/sg['engagement_score'].mean()*100-100:+.1f}% engagement vs average sentiment, watch time {sg.loc[s,C['watch']]:.1f}s, comments {sg.loc[s,C['comments']]:.2f}.")

for k, v in L.items():
    print(k.upper())
    for line in v: print("  -", line)

## One-page Summary (auto-generated from the results above)

In [ ]:
from IPython.display import Markdown, display

summary = [f"# Social Media Engagement Analytics - Summary",
           f"**Data:** {len(raw):,} posts loaded, {len(df):,} after cleaning ({len(raw)-len(df)} duplicates/invalid rows removed). "
           f"Period: {df[C['date']].min():%b %Y} to {df[C['date']].max():%b %Y}. Engagement score = likes + 2 x comments + 3 x shares.", ""]
for k, v in L.items():
    summary.append(f"## {k}")
    summary += [f"- {x}" for x in v]
    summary.append("")
summary += ["## Recommendations",
            f"- Prioritise **{t_hi}** content and the **{c_hi}** category; test more in under-performing formats before cutting them.",
            f"- Schedule key posts in the **{tod.idxmax()}** window to maximise impressions.",
            f"- Target audiences in **{k_hi}** and the **{ag.idxmax()}** age group where engagement rate is highest.",
            "- Caveat: patterns are associations in observational data; differences should be tested for significance before acting."]
summary_md = "\n".join(summary)
display(Markdown(summary_md))
open("summary.md", "w").write(summary_md)

In [ ]:
# Export the summary as a Word document (one page) - run in Colab, file appears in the left Files panel
try:
    import docx
except ImportError:
    !pip -q install python-docx
    import docx
from docx.shared import Pt, Inches
d = docx.Document()
for s in d.sections:
    s.top_margin = s.bottom_margin = Inches(.6); s.left_margin = s.right_margin = Inches(.7)
d.styles["Normal"].font.size = Pt(9.5)
for line in summary_md.splitlines():
    if line.startswith("# "):   d.add_heading(line[2:], 1)
    elif line.startswith("## "): d.add_heading(line[3:], 2)
    elif line.startswith("- "):  d.add_paragraph(line[2:].replace("**", ""), style="List Bullet")
    elif line.strip():           d.add_paragraph(line.replace("**", ""))
d.save("Social_Media_Engagement_Summary.docx")
print("Saved Social_Media_Engagement_Summary.docx")